# 2. Multi-Objective Optimization — robust MOMILP + AUGMECON2

Bước 2 của chuỗi **Robust → Multi-Objective Optimization → TOPSIS**.

- **Đọc:** `Model - Data used/` (kho, chi phí, DOS, lead time) và `Model Outputs/demand_robust.parquet` (từ notebook 1).
- **Mô hình:** MILP chọn mở/đóng kho (`Y_j`) và luồng hàng; cầu robust Bertsimas & Sim với ngân sách `Γ`.
- **Mục tiêu:** `f1` tổng chi phí (min), `f2` cầu giao ngoài chuẩn lead time `L_max` (min), `f3` CO2 (min).
- **Giải:** Pyomo + HiGHS (`appsi_highs`), AUGMECON2 → tập Pareto.
- **Ghi:** `Model Outputs/pareto.parquet` → notebook 3 đọc.

**Thứ tự xây (Decision_Log 1.6):** `f1` một mình (Γ = 0) → kiểm tra → thêm robust → thêm `f2`, `f3` → payoff table → AUGMECON2.

In [ ]:
# ===== SETUP =====
import pandas as pd                       # xử lý bảng
import numpy as np                        # tính toán
import pyomo.environ as pyo               # xây mô hình tối ưu
from pathlib import Path                  # ghép đường dẫn

pd.set_option("display.width", 200)

IN = Path("../Model - Data used")         # data đã clean (chỉ đọc)
OUT = Path("../Model Outputs")            # kết quả của các notebook model

fac = pd.read_parquet(IN / "facility_master.parquet")                    # 53 kho: chi phí cố định, handling, capacity
dos = pd.read_parquet(IN / "dos.parquet")                                # DOS mỗi kho (ngày)
tr = pd.read_parquet(IN / "cost_transfer.parquet")                       # 369 tuyến kho -> kho, IDR/pallet (2025)
lm = pd.read_parquet(IN / "cost_lastmile_b2b.parquet")                   # 14.770 tuyến kho -> nhóm, IDR/pallet (2025)
lead = pd.read_parquet(IN / "lead_time.parquet")                         # lead time + khoảng cách mỗi tuyến
demand = pd.read_parquet(OUT / "demand_robust.parquet")                  # d̄, d̂ mỗi nhóm (notebook 1)
print(f"Kho: {len(fac)} | tuyến transfer: {len(tr)} | tuyến last-mile: {len(lm):,} | nhóm: {len(demand):,}")

## 2.1 Tham số + kiểm tra capacity so với cầu

*(chưa viết)*

## 2.2 `f1` — MILP chi phí, tất định (Γ = 0)

*(chưa viết)*

## 2.3 Robust: thêm ràng buộc Bertsimas & Sim, quét `Γ`

*(chưa viết)*

## 2.4 `f2` — độ phủ lead time

Nhóm k được phủ bởi kho j nếu `LT_jk ≤ L_max` (đảo của k). `L_max` đang thử theo SLA công ty: Java 1 ngày, ngoài Java 3 ngày (Decision_Log 3.4 chưa chốt). Cell dưới: tập kho phủ `N_k` và mức phủ tối đa khi mở hết kho.

In [ ]:
# ===== 2.4a ĐỘ PHỦ: tập kho phủ N_k và mức phủ tối đa có thể đạt =====
L_MAX = {"Java": 1.0}                                                     # đảo có ngưỡng riêng
L_MAX_KHAC = 3.0                                                          # mọi đảo còn lại

# (1) Đảo của mỗi nhóm khách: lấy đảo của các ship-to trong nhóm (mỗi nhóm chỉ thuộc 1 đảo, Decision_Log 4.6)
mp = pd.read_parquet(IN / "shipto_group_map.parquet")
cm = pd.read_parquet(IN / "customer_master.parquet", columns=["LocationID", "MainIsland"])
dao_nhom = (mp.merge(cm, left_on="ShipToID", right_on="LocationID")
              .groupby("ShipToGroupID")["MainIsland"].agg(lambda s: s.mode().iat[0]))
G = set(demand["ShipToGroupID"])
assert set(dao_nhom.index) == G                                           # nhóm nào cũng có đảo

# (2) Lead time các tuyến last-mile + ngưỡng của nhóm đích
lm_lt = lead[lead["Loai"] == "LastMile"][["OriginID", "DestinationID", "TotalLeadTimeInDays"]].copy()
lm_lt["DaoNhom"] = lm_lt["DestinationID"].map(dao_nhom)
lm_lt["L_max"] = lm_lt["DaoNhom"].map(L_MAX).fillna(L_MAX_KHAC)           # ngưỡng theo đảo
lm_lt["Phu"] = lm_lt["TotalLeadTimeInDays"] <= lm_lt["L_max"]              # True = kho j phủ được nhóm k

# (3) N_k = số kho phủ được mỗi nhóm
so_kho_phu = lm_lt[lm_lt["Phu"]].groupby("DestinationID").size().reindex(sorted(G), fill_value=0)
d = demand.set_index("ShipToGroupID")["d_bar"]
print("Số kho phủ được mỗi nhóm (số kho: số nhóm):", so_kho_phu.value_counts().sort_index().to_dict())

# (4) Mức phủ tối đa: nếu MỞ HẾT 53 kho thì bao nhiêu % cầu được phủ (cận trên của dịch vụ)
bang = pd.DataFrame({"Dao": dao_nhom, "d_bar": d, "CoTheDuocPhu": so_kho_phu > 0})
tom = bang.groupby("Dao").apply(lambda x: pd.Series({
    "so_nhom": len(x),
    "nhom_khong_the_phu": int((~x["CoTheDuocPhu"]).sum()),
    "pct_cau_phu_toi_da": x.loc[x["CoTheDuocPhu"], "d_bar"].sum() / x["d_bar"].sum() * 100}))
print("\nMở hết kho thì phủ được tối đa (theo đảo):")
print(tom.round(1))
print(f"\nToàn quốc: phủ tối đa {bang.loc[bang['CoTheDuocPhu'], 'd_bar'].sum() / bang['d_bar'].sum() * 100:.1f}% cầu | "
      f"{int((~bang['CoTheDuocPhu']).sum())} nhóm không kho nào phủ được")

# (5) Độ nhạy theo L_max: % cầu có ít nhất 1 kho trong ngưỡng (để biết ngưỡng có tạo trade-off không)
best_lt = lm_lt.groupby("DestinationID")["TotalLeadTimeInDays"].min()     # lead time nhanh nhất có thể của mỗi nhóm
print("\n% cầu có ít nhất 1 kho trong ngưỡng (một ngưỡng chung cho cả nước):")
for L in [0.25, 0.5, 1, 2, 3, 5]:
    print(f"   L_max = {L:>4} ngày: {d[best_lt.reindex(d.index) <= L].sum() / d.sum() * 100:5.1f}%")

## 2.5 `f3` — CO2 (GLEC)

*(chưa viết)*

## 2.6 Payoff table + AUGMECON2 → tập Pareto

*(chưa viết)*